**Import**

In [ ]:
!pip install pybullet
import pybullet as p

In [ ]:
import os 
import pybullet_data
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
from PIL import Image
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms
from pathlib import Path
from torch.distributions import Normal
import math
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import BasePolicy
from stable_baselines3.common.policies import ActorCriticPolicy
import gym
from sklearn.model_selection import train_test_split, random_split
import matplotlib.pyplot as plt
import csv 

In [ ]:
!git clone https://github.com/Mvryo02/RL_project2

**Classes and functions definitions to get the images from Pybullet and the state**

In [ ]:
def get_camera_settings():
    width, height = 480, 480
    distance = 5  # Camera distance from the plane
    angle = math.radians(90)  # angle bewteen camera and plane

    # Coordinates for camera
    camera_x = 0
    camera_y = -distance * math.cos(angle)  
    camera_z = distance * math.sin(angle)  

    view_matrix = p.computeViewMatrix(
        cameraEyePosition=[camera_x, camera_y, camera_z],
        cameraTargetPosition=[0, 0, 0],  
        cameraUpVector=[0, 0, 1]  
    )
    projection_matrix = p.computeProjectionMatrixFOV(
        fov=90, aspect=float(width) / height, nearVal=0.1, farVal=100.0
    )
    return width, height, view_matrix, projection_matrix

In [ ]:
def get_image(robot, view_matrix, projection_matrix, width=480, height=480):
    
    #_, current_orientation = p.getBasePositionAndOrientation(robot)

    rgb_img, _, _ = p.getCameraImage(
        width, height, viewMatrix=view_matrix, projectionMatrix=projection_matrix
    )[2:5]

    rgb_img = np.array(rgb_img)  # Convert to np array
    rgb_img = rgb_img[:, :, :3]  # It consider only the first three channels RGB
    
    return rgb_img

In [ ]:
def draw_filled_square(center, size, color=[1, 0, 0, 1]):
    visual_shape = p.createVisualShape(
        shapeType=p.GEOM_BOX,
        halfExtents=[size / 2, size / 2, 0.01],
        rgbaColor=color
    )
    square_id = p.createMultiBody(
        baseVisualShapeIndex=visual_shape,
        basePosition=center
    )
    return square_id

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()

        
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=3, padding=1),  
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=5, stride=3, padding=1),  # 64x53x53
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=5, stride=3, padding=1),  # 128x17x17
            nn.ReLU(),
            nn.Flatten(),  
            nn.Linear(128*17*17, 64),  
            nn.ReLU()
        )

        
        self.decoder = nn.Sequential(
            nn.Linear(64, 128*17*17),  
            nn.ReLU(),
            nn.Unflatten(1, (128, 17, 17)),  
            nn.ConvTranspose2d(128, 64, kernel_size=7, stride=3, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=5, stride=3, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=7, stride=3, padding=1, output_padding= 1),
            nn.Sigmoid()  
        )

    def forward(self, x):
        encoded = self.encoder(x)
        
        decoded = self.decoder(encoded)
        return decoded

**Training autoencoder**

In [ ]:
p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())  
p.setGravity(0, 0, -9.81)  
plane_id = p.loadURDF("plane.urdf") #to uplaod the basic plane 
robot_start_pos = [0, 0, 0.5]
robot_start_orientation = p.getQuaternionFromEuler([0, 0, 0])

robot = p.loadURDF("husky/husky.urdf", robot_start_pos, robot_start_orientation)

In [ ]:
def get_random_position(range_min=-5, range_max=5):
    
    return [random.uniform(range_min, range_max), random.uniform(range_min, range_max), 0.01]

def collect_data(robot, num_steps=10000, save_dir="dataset"):
    if not os.path.exists(save_dir):  
        os.makedirs(save_dir)

    width, height, view_matrix, projection_matrix = get_camera_settings()
    data = []
    
    square_id = None  
    for step in range(num_steps):
        
        # Change position every 200 steps
        if step % 200 == 0:
            if square_id is not None:
                p.removeBody(square_id)  
            square_position = get_random_position()
            square_id = draw_filled_square(square_position, size=0.8)

        
        rgb_img, _, _ = p.getCameraImage(
            width, height, viewMatrix=view_matrix, projectionMatrix=projection_matrix
        )[2:5]

        rgb_img = np.array(rgb_img)[:, :, :3]  
        img_path_new = os.path.join(save_dir, f"obs_{step}.png")
        Image.fromarray(rgb_img).save(img_path_new, format='PNG')

        # Random action
        linear_velocity = random.uniform(-2, 2)
        angular_velocity = random.uniform(-1, 1)
        p.resetBaseVelocity(robot, linearVelocity=[linear_velocity, 0, 0], angularVelocity=[0, 0, angular_velocity])
        p.stepSimulation()

        
        if step > 1:
            data.append({
                "step": step,
                "action": [linear_velocity, angular_velocity],
                "observation": image_path,
                "next_observation": img_path_new
            })
        image_path = img_path_new

        if step % 100 == 0:
            print(f"Step {step}, robot position: {p.getBasePositionAndOrientation(robot)}")

    with open('dataset.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=["step", "action", "observation", "next_observation"])
        writer.writeheader()
        writer.writerows(data)

    dataset_path = os.path.join(save_dir, "dataset.npy")
    np.save(dataset_path, data)
    print(f"Dataset saved in {dataset_path}")

#collect_data(robot)

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, dataset_path, transform=None):
        self.dataset_path = Path(dataset_path)
        self.image_paths = list(self.dataset_path.glob("*.png"))  
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")  
        if self.transform:
            image = self.transform(image)  
        return image  
transform = transforms.Compose([
    transforms.Resize((480, 480)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [ ]:
dataset_path = '/kaggle/working/dataset'


dataset = ImageDataset(dataset_path, transform=transform)
train_percentage = 0.7
val_percentage = 0.15
test_percentage = 0.15

train_size = int(len(dataset) * train_percentage)
val_size = int(dataset.__len__() * val_percentage)
test_size = dataset.__len__() - train_size - val_size

train_data, val_data, test_data = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_data, batch_size=37, shuffle=False)
val_loader = DataLoader(val_data, batch_size=37, shuffle=False)
test_loader = DataLoader(test_data, batch_size=37, shuffle=False)

In [ ]:
Loss = []
Validation_Loss = []

def train_autoencoder(model, train_loader,val_loader,  criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for images in train_loader:
            images = images.cuda()  
            optimizer.zero_grad()      
            outputs = model(images)    
            loss = criterion(outputs, images) 
            loss.backward()            
            optimizer.step()           
            total_loss += loss.item()

        val_loss = 0.0
        model.eval()
        for images in val_loader:
            images = images.cuda()
            val_predictions = model(images)

           
            val_loss += criterion(val_predictions, images).item()
            

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}, Validation Loss: {val_loss/len(val_loader):.4f}")

model2 = Autoencoder()
model2.cuda()
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model2.parameters(), lr= 0.001)


train_autoencoder(model2, train_loader, val_loader, criterion, optimizer, epochs=30)

In [ ]:
def test_autoencoder(autoencoder, test_loader, criterion, device, visualize=False):
    
    autoencoder.eval()  
    test_loss = 0.0
    images_original = []
    images_reconstructed = []
    
    with torch.no_grad():  
        for batch in test_loader:
           
            images = batch.to(device)
            
            
            reconstructed_images = autoencoder(images)
            
            
            loss = criterion(reconstructed_images, images)
            test_loss += loss.item()
            
            
            if visualize and len(images_original) < 5:  
                images_original.append(images[0].cpu())  
                images_reconstructed.append(reconstructed_images[0].cpu()) 

   
    test_loss /= len(test_loader)
    
    
    return test_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_loss = test_autoencoder(model2, test_loader, criterion, device, visualize=True)

print(f"Test Reconstruction Loss: {test_loss:.4f}")

In [ ]:
torch.save(model2.state_dict(), 'autoencoder.pth')

**Environment definition for the first task**

In [ ]:
class RobotEnv(gym.Env):
    def __init__(self):
        self.render_mode = "rgb_array"
        self.steps = 0
        self.dataset_step = 0
        self.square_id = None  
        self.total_reward = 0
        self.physics_client = p.connect(p.DIRECT)  
        p.setAdditionalSearchPath(pybullet_data.getDataPath())  
        p.setGravity(0, 0, -9.81) 
        self.targets = []
        self.plane_id = p.loadURDF("plane.urdf")
        self.robot_start_pos = [0, 0, 0.5]
        self.robot_start_orientation = p.getQuaternionFromEuler([0, 0, 0])
        self.robot = p.loadURDF("husky/husky.urdf", self.robot_start_pos, self.robot_start_orientation) 
        p.configureDebugVisualizer(p.COV_ENABLE_RENDERING, 1)
        dt = 0.2
        p.setTimeStep(dt)
        center = [0, 0, 0.1]  
        radius =  5         
        self.autoencoder = Autoencoder().cuda()
        self.autoencoder.load_state_dict(torch.load("/kaggle/working/RL_project2/autoencoder.pth"))
        self.autoencoder.eval()  
        self.encoder = self.autoencoder.encoder
        self.width, self.height, self.view_matrix, self.projection_matrix = get_camera_settings()

        # state and action spaces
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(64,), dtype=np.float32)
        self.action_space = gym.spaces.Box(low=np.array([-1.0, -np.pi]), high=np.array([2.0, np.pi]), dtype=np.float32)

        #uncomment this to test teacher and students
        #for i in range(20):
         #   self.targets.append([random.uniform(-2, 2), random.uniform(-2, 2), 0.1]) 
        #self.target_index = 0
        
        self.reset()
        
    def reset(self):
        
        self.steps = 0
        self.total_reward = 0

        #reset velocity and position for the robot
        p.resetBaseVelocity(self.robot, linearVelocity=[0, 0, 0], angularVelocity=[0, 0, 0])

        p.resetBasePositionAndOrientation(self.robot, self.robot_start_pos, self.robot_start_orientation)

        # if there is a square on the scene, it is deleted 
        if self.square_id is not None:
            p.removeBody(self.square_id)

        # add a new target
        target_position = [random.uniform(-2, 2), random.uniform(-2, 2), 0.1]  #comment for test
        #target_position = self.targets[self.target_index]  #for test
        self.square_id = draw_filled_square(target_position, 0.5)  
        self.target_position = np.array(target_position[:2])
        #self.target_index = (self.target_index+1)%20 #for test
        print("Position target:", self.target_position)

        return self._get_state()
        
    def step(self, action):
        self.steps += 1
        v, w = action
        dt = 0.2
        
        #eevaluate the old distance
        pos, orn = p.getBasePositionAndOrientation(self.robot)
        old_distance = np.linalg.norm(pos[:2] - self.target_position)
        
        p.resetBaseVelocity(self.robot, linearVelocity=[v, 0, 0], angularVelocity=[0, 0, w])
        p.stepSimulation()
        position, orientation = p.getBasePositionAndOrientation(self.robot)
       
        
        # Evaluation current distance
        new_distance = np.linalg.norm(position[:2] - self.target_position)
        delta_y = abs(position[1] - self.target_position[1]) - abs(pos[1] - self.target_position[1])
        
        
        reward = (old_distance-new_distance)*100-100*delta_y #
        
        #if abs(position[1]-pos[1])< 0.01: #this is an additional reward term to 
         #   reward -= 1
        
        ray = math.sqrt(position[0]*position[0]+position[1]*position[1])
        done = False
        if new_distance < 0.3:  
            reward += 200  
            print("Target point reached!")
            done = True
        elif self.steps >= 400:  
            done = True
        if ray >= 4:
            reward -= 100  
            
        self.total_reward += reward
        if self.steps % 100 == 0:
            print(f"Step {self.steps}, Mean Reward: {self.total_reward/self.steps:.2f}")
        
        return self._get_state(), reward, done, {}
        
    def render(self):
        
        view_matrix = self.view_matrix
        projection_matrix = self.projection_matrix
        return get_image(robot = self.robot, view_matrix = self.view_matrix, projection_matrix = self.projection_matrix)
        
    def _get_state(self):
        rgb_img = get_image(robot=self.robot, view_matrix=self.view_matrix, projection_matrix=self.projection_matrix)
        image = Image.fromarray(rgb_img)
        img_path_new = os.path.join("dataset", f"obs_{self.dataset_step}.png") 
        if self.steps%2000 == 0:
            image.save(img_path_new, format='PNG')
        rgb_tensor = torch.tensor(rgb_img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).cuda()
        self.dataset_step += 1
        with torch.no_grad():
            encoded_state = self.encoder(rgb_tensor).cpu()
        
        return encoded_state.numpy()

**Environment definition for the second task**

In [ ]:
class RobotEnvTC(gym.Env):
    def __init__(self):
        self.steps = 0
        self.dataset_step = 0
        self.total_reward = 0
        self.k = 5
        self.position_history = []
        self.last_position = None  
        self.square_pos = None
        self.square_id = None
        self.circle_radius = 0.5
        self.circle_center = np.array([0.0, 0.0])
        self.targets = []
        self.physics_client = p.connect(p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())  
        p.setGravity(0, 0, -9.81)
        self.plane_id = p.loadURDF("plane.urdf")
        
        self.robot_start_pos = [0, 0, 0.5]  
        self.robot_start_orientation = p.getQuaternionFromEuler([0, 0, 0])
        self.robot = p.loadURDF("husky/husky.urdf", self.robot_start_pos, self.robot_start_orientation)  
        p.configureDebugVisualizer(p.COV_ENABLE_RENDERING, 1)
        dt = 0.2
        p.setTimeStep(dt)
        

        
        self.autoencoder = Autoencoder().cuda()
        self.autoencoder.load_state_dict(torch.load("/kaggle/working/RL_project2/autoencoder.pth"))
        self.autoencoder.eval()
        self.encoder = self.autoencoder.encoder
        self.width, self.height, self.view_matrix, self.projection_matrix = get_camera_settings()

        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(64,), dtype=np.float32)
        self.action_space = gym.spaces.Box(low=np.array([-1.0, -np.pi]), high=np.array([2.0, np.pi]), dtype=np.float32)
        #uncomment this to test teacher and students
        #for i in range(20):
         #   self.targets.append([random.uniform(-2, 2), random.uniform(-2, 2), 0.1]) 
        #self.target_index = 0

        self.reset()

    def reset(self):
        self.steps = 0
        self.total_reward = 0
        self.position_history = []
        p.resetBaseVelocity(self.robot, linearVelocity=[0, 0, 0], angularVelocity=[0, 0, 0])
        p.resetBasePositionAndOrientation(self.robot, self.robot_start_pos, self.robot_start_orientation)
        if self.square_id is not None:
            p.removeBody(self.square_id)
        target_position = [random.uniform(-2, 2), random.uniform(-2, 2), 0.1]  
        #target_position = self.targets[self.target_index]  #for test
        self.square_id = draw_filled_square(target_position, 0.5, color = [0,0,1,1])  
        self.square_pos = np.array(target_position[:2])
        #self.target_index = (self.target_index+1)%20 #for test 
        #self.last_position = np.array(self.robot_start_pos[:2])  
        print("Position target:", self.square_pos)
        return self._get_state()

    def step(self, action):
        self.steps += 1
        v, w = action
        
       
        pos, orn = p.getBasePositionAndOrientation(self.robot)
        p.resetBaseVelocity(self.robot, linearVelocity=[v, 0, 0], angularVelocity=[0, 0, w])
        p.stepSimulation()
        new_pos, _ = p.getBasePositionAndOrientation(self.robot)
        new_pos = np.array(new_pos[:2])
        
        
        lambda_coeff = 10
        
        
        
        distance_from_circle = abs(np.linalg.norm(new_pos - self.square_pos) - self.circle_radius)
        reward_circle = lambda_coeff * (1 - lambda_coeff * (distance_from_circle ** 2))

        
        self.position_history.append(new_pos[:2])
        if len(self.position_history) > self.k + 1:
            self.position_history.pop(0)
            
        if len(self.position_history) == self.k + 1:
            movement_bonus = np.linalg.norm(self.position_history[-1] - self.position_history[0])
        else:
            movement_bonus = 0
            
        # Penalty if the robot exit a certain area
        if np.linalg.norm(new_pos - self.circle_center) >= 4:
            penalty = -lambda_coeff*lambda_coeff
        else:
            penalty = 0
        
        # final reward
        reward = reward_circle*movement_bonus + penalty
        self.total_reward += reward

       
        done = self.steps >= 400  # The ep ends only if 400 steps
        if self.steps% 100 == 0:
            print("total reward:", self.total_reward, "mean reward", self.total_reward/self.steps)
        
        #self.last_position = new_pos

        return self._get_state(), reward, done, {}

    def _get_state(self):
        rgb_img = get_image(robot=self.robot, view_matrix=self.view_matrix, projection_matrix=self.projection_matrix)
        image = Image.fromarray(rgb_img)
        img_path_new = os.path.join("dataset", f"obs_{self.dataset_step}.png") 
        if self.steps%2000 == 0:
            image.save(img_path_new, format='PNG')
        rgb_tensor = torch.tensor(rgb_img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).cuda()
        self.dataset_step += 1
        with torch.no_grad():
            encoded_state = self.encoder(rgb_tensor).cpu()
        
        return encoded_state.numpy()

**Training**

In [ ]:
os.makedirs("dataset", exist_ok=True)

In [ ]:
env = RobotEnv()
env = DummyVecEnv([lambda: env])
#model = PPO("MlpPolicy", env, verbose =1, ent_coef = 0.02)
model = PPO.load("/kaggle/working/RL_project2/ppo_task1.zip", env=env, verbose = 2)  
model.learn(total_timesteps=100000, log_interval= 1)
model.save("ppo_task1.zip")

In [ ]:
env = RobotEnvTC()
env = DummyVecEnv([lambda: env])
#model = PPO("MlpPolicy", env, verbose=1, ent_coef=0.02)
model = PPO.load("/kaggle/working/RL_project2/ppo_task2", env = env)
model.learn(total_timesteps=400000)  
model.save("ppo_task2.zip")

**Testing**

In [ ]:
env = RobotEnv()
env = DummyVecEnv([lambda: env])
model = PPO.load("/kaggle/working/RL_project2/ppo_task1", env = env)

num_episodes = 10  # number of episodes
num_step = 400
rewards = []  # This list contains all the rewards

for ep in range(num_episodes):
    obs = env.reset()
    done = False
    episode_reward = 0
    #positions = []  
    step = 0
    while step < num_step:
        step += 1
        action, _states = model.predict(obs, deterministic=True) 
        obs, reward, done, info = env.step(action)
        episode_reward += reward
        
    
    rewards.append(episode_reward)
    print(f"Episode {ep+1}: Total Reward: {episode_reward}")

env.close()  

In [ ]:
env = RobotEnvTC()
env = DummyVecEnv([lambda: env])
model = PPO.load("/kaggle/working/RL_project2/ppo_task2", env = env)

num_episodes = 10  # number of episodes
num_step = 400
rewards = []  # This list contains all the rewards

for ep in range(num_episodes):
    obs = env.reset()
    done = False
    episode_reward = 0
    #positions = []  
    step = 0
    while step < num_step:
        step += 1
        action, _states = model.predict(obs, deterministic=True) 
        obs, reward, done, info = env.step(action)
        episode_reward += reward
                
    
    rewards.append(episode_reward)
    print(f"Episode {ep+1}: Total Reward: {episode_reward}")

env.close()  

**Define the Distillation datasets**

In [ ]:
obs_list = []
means_list = []
stds_list = []

env = RobotEnv()
env = DummyVecEnv([lambda: env])
teacher_model = PPO.load("/kaggle/working/RL_project2/ppo_task1.zip", env = env)

num_samples = 250000  

obs = env.reset()

for _ in range(num_samples):
    obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).cuda()    
    
    action_distribution = teacher_model.policy.get_distribution(obs_tensor)

    # Get mean and standard deviation from the action distribution
    action_mean = action_distribution.distribution.mean.cpu().detach().numpy().flatten()
    action_std = action_distribution.distribution.stddev.cpu().detach().numpy().flatten()
    # add data to the lists
    obs_list.append(obs)
    means_list.append(action_mean)
    stds_list.append(action_std)
    
    
    action = action_mean  # chose the mean as action
    obs, _, done, _ = env.step(action.reshape(1, -1))
    
    


obs_array = np.array(obs_list)
means_array = np.array(means_list)
stds_array = np.array(stds_list)


np.savez("distillation_dataset_task1.npz", obs=obs_array, teacher_means=means_array, teacher_stds=stds_array)

In [ ]:
obs_list = []
means_list = []
stds_list = []

env = RobotEnvTC()
env = DummyVecEnv([lambda: env])
teacher_model = PPO.load("/kaggle/working/RL_project2/ppo_task2.zip", env = env)

num_samples = 250000  

obs = env.reset()

for _ in range(num_samples):
    obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).cuda()
    
    
    action_distribution = teacher_model.policy.get_distribution(obs_tensor)

    # Get mean and standard deviation from the action distribution
    action_mean = action_distribution.distribution.mean.cpu().detach().numpy().flatten()
    action_std = action_distribution.distribution.stddev.cpu().detach().numpy().flatten()
    # add data to the lists
    obs_list.append(obs)
    means_list.append(action_mean)
    stds_list.append(action_std)
    
    
    action = action_mean  # chose the mean as action
    obs, _, done, _ = env.step(action.reshape(1, -1))

obs_array = np.array(obs_list)
means_array = np.array(means_list)
stds_array = np.array(stds_list)


np.savez("distillation_dataset_task2.npz", obs=obs_array, teacher_means=means_array, teacher_stds=stds_array)

**training student models**

In [ ]:
class StudentPolicy(nn.Module):
    def __init__(self, input_dim=64, output_dim=2):
        super(StudentPolicy, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)  # 64 → 128
        self.fc2 = nn.Linear(128, 64)         # 128 → 64
        self.fc3 = nn.Linear(64, 32)          # 64 → 32

        # Output layers
        self.mean = nn.Linear(32, output_dim)  #Mean Output
        self.log_var = nn.Linear(32, output_dim)  #standard deviation

        self.relu = nn.ReLU()

    def forward(self, x):
        
        x.squeeze(1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))

        mean = self.mean(x)  
        log_var = self.log_var(x)  

        return mean, log_var  

In [ ]:
def kl_divergence_loss(student_logits, teacher_logits, tau):
    teacher_probs = F.softmax(teacher_logits / tau, dim=-1)  
    student_log_probs = F.log_softmax(student_logits / tau, dim=-1) 
    return F.kl_div(student_log_probs, teacher_probs, reduction="batchmean")


In [ ]:
data = np.load("/kaggle/working/RL_project2/distillation_dataset_task1.npz") #or distillatio_dataset_task2 if the stuent should learn the 2nd task


obs = data["obs"]  # state
teacher_means = data["teacher_means"]  # mean for each state
teacher_stds = data["teacher_stds"]    # standard deviation for each state 

#80% train, 10% validation, 10% test
obs_train, obs_temp, means_train, means_temp, stds_train, stds_temp = train_test_split(
    obs, teacher_means, teacher_stds, test_size=0.2, random_state=42
)
obs_val, obs_test, means_val, means_test, stds_val, stds_test = train_test_split(
    obs_temp, means_temp, stds_temp, test_size=0.5, random_state=42
)


In [ ]:
obs_val = torch.from_numpy(obs_val)
means_val = torch.from_numpy(means_val)
stds_val = torch.from_numpy(stds_val)
obs_train = torch.from_numpy(obs_train)
means_train = torch.from_numpy(means_train)
stds_train = torch.from_numpy(stds_train)
obs_train = torch.tensor(obs_train, dtype=torch.float32)
means_train = torch.tensor(means_train, dtype=torch.float32)
stds_train = torch.tensor(stds_train, dtype=torch.float32)
obs_val = torch.tensor(obs_val, dtype=torch.float32)
means_val = torch.tensor(means_val, dtype=torch.float32)
stds_val = torch.tensor(stds_val, dtype=torch.float32)

batch_size = 64
num_epochs = 1000
learning_rate = 0.0003

# Dataset di training
train_data = TensorDataset(obs_train, means_train, stds_train)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

val_data = TensorDataset(obs_val, means_val, stds_val)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=True)

student_model = StudentPolicy()
#uncomment to train a model which is already trained for another task
#student_model.load_state_dict(torch.load("/kaggle/working/RL_project2/student_policy_task_1e2_KL_1.pth"))

student_model.cuda()

optimizer = optim.Adam(student_model.parameters(), lr=learning_rate)


#mse_loss = nn.MSELoss()

# Training loop
for epoch in range(num_epochs):
    student_model.train()  
    epoch_loss = 0.0
    tau = 1
    
    for obs_batch, means_batch, stds_batch in train_loader:
        
        obs_batch = obs_batch.float().cuda()
        means_batch = means_batch.cuda()
        stds_batch = stds_batch.cuda()
        
        student_logits_mean, student_logits_std = student_model(obs_batch)

        #loss_mean = mse_loss(student_logits_mean, means_batch) #uncomment this if the loss is MSE
        #loss_std = mse_loss(student_logits_std, stds_batch)
        
        loss_mean = kl_divergence_loss(student_logits_mean, means_batch, tau) #comment this if loss is MSE
        loss_std = kl_divergence_loss(student_logits_std, stds_batch, tau)
        
        loss = loss_mean + loss_std
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()


    epoch_loss /= len(train_loader)
    print(f"Epoch {epoch}/{num_epochs} - Training Loss: {epoch_loss:.4f}")

    if (epoch+1) % 100  == 0:
        student_model.eval()  
        total_loss = 0.0
        num_batches = 0

        with torch.no_grad():
            for obs_batch, means_batch, stds_batch in val_loader:
                obs_batch = obs_batch.float().cuda()
                means_batch = means_batch.cuda()
                stds_batch = stds_batch.cuda()

                student_logits_mean, student_logits_std = student_model(obs_batch)

                #loss_mean = mse_loss(student_logits_mean, means_batch)
                #loss_std = mse_loss(student_logits_std, stds_batch)
                
                loss_mean = kl_divergence_loss(student_logits_mean, means_batch, tau)
                loss_std = kl_divergence_loss(student_logits_std, stds_batch, tau)

                loss += loss_mean.item() + loss_std.item()
        
                total_loss += loss.item()
                num_batches += 1

        avg_loss = total_loss / num_batches
        print(f"Validation Loss: {avg_loss:.4f}")

        torch.save(student_model.state_dict(), "student_policy_task_1e2_KL_1.pth")


In [ ]:
obs_test = torch.from_numpy(obs_test)
means_test = torch.from_numpy(means_test)
stds_test = torch.from_numpy(stds_test)
obs_test = torch.tensor(obs_test, dtype=torch.float32)
means_test = torch.tensor(means_test, dtype=torch.float32)
stds_test = torch.tensor(stds_test, dtype=torch.float32)

test_data = TensorDataset(obs_test, means_test, stds_test)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

tau = 0.1
total_loss =0.0
num_batches = 0
with torch.no_grad():
    for obs_batch, means_batch, stds_batch in test_loader:
        loss = 0.0
        obs_batch = obs_batch.float().cuda()
        means_batch = means_batch.cuda()
        stds_batch = stds_batch.cuda()

        student_logits_mean, student_logits_std = student_model(obs_batch)

        #loss_mean = mse_loss(student_logits_mean, means_batch)
        #loss_std = mse_loss(student_logits_std, stds_batch)
        
        loss_mean = kl_divergence_loss(student_logits_mean, means_batch, tau)
        loss_std = kl_divergence_loss(student_logits_std, stds_batch, tau)


        total_loss += loss_mean.item() + loss_std.item()
        num_batches += 1


avg_loss = total_loss / num_batches
print(f"Validation Loss: {avg_loss:.4f}")


**Testing student models**

In [ ]:
#to test the student on the first task
env = RobotEnv()
env = DummyVecEnv([lambda: env])
teacher_model = PPO.load("/kaggle/working/RL_project2/ppo_task1.zip", env = env) #or ppo_task2 if the student should learn the second task

In [ ]:
#to test the student on the second task
env = RobotEnvTC()
env = DummyVecEnv([lambda: env])
teacher_model = PPO.load("/kaggle/working/RL_project2/ppo_task2.zip", env = env)

In [ ]:
#for the first task both student_model_1 and student_model_1e2 are taken into account, for 2nd task only student_model_1e2

#student_model_1 = StudentPolicy()  
#student_model_1.load_state_dict(torch.load("/kaggle/working/RL_project2/student_policy_task_1.pth"))
#student_model_1.cuda()
student_model_1e2 = StudentPolicy()  
student_model_1e2.load_state_dict(torch.load("/kaggle/working/RL_project2/student_policy_task_1e2.pth"))
student_model_1e2.cuda()

In [ ]:
total_rewards = []
def evaluate_policy(model, env, num_episodes=20, seed=42, is_student=False):

    np.random.seed(seed)
    env.seed(seed)
    
    for episode in range(num_episodes):
        obs = env.envs[0]._get_state()  
        episode_reward = 0
        done = False
        steps = 0
        
        while not done:
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).cuda()
            
            with torch.no_grad():
                if is_student:  # Student Model
                    action_mean, _ = model(obs_tensor)  # It returns mean and standard deviation, just use the mean as action
                else:  # Teacher PPO Model
                    latent_pi, _ = model.policy.mlp_extractor(obs_tensor)
                    action_mean = model.policy.action_net(latent_pi) 

                action_mean = action_mean.cpu().numpy().flatten()  

            obs, reward, done, info = env.step(action_mean.reshape(1, -1))
            
            episode_reward += reward
            

        total_rewards.append(episode_reward)
        
        print("Reward episode: ", episode_reward)
    avg_reward = np.mean(total_rewards)
    
    
    print(f"{'Student' if is_student else 'Teacher'} Policy - Avg Reward: {avg_reward:.4f}") 
    return avg_reward 


In [ ]:
teacher_reward = evaluate_policy(teacher_model, env)
#student_reward_1 = evaluate_policy(student_model_1, env, is_student = True)
student_reward_1e2  = evaluate_policy(student_model_1e2, env, is_student = True)

#print(f"Teacher Reward: {teacher_reward:.4f}, Student 1 Reward: {student_reward_1:.4f},  Student 2 Reward: {student_reward_1e2:.4f} ") #if 2 students
print(f"Teacher Reward: {teacher_reward:.4f},  Student 2 Reward: {student_reward_1e2:.4f} ") #if 1 student


In [ ]:
total_rewards = np.array(total_rewards)

#teacher, student1, student2 = np.split(total_reward, 3) #for first task
teacher, student2 = np.split(total_rewards, 2)

mean_teacher = np.mean(teacher)
var_teacher = np.var(teacher)

#mean_student1 = np.mean(student1)
#var_student1 = np.var(student1)

mean_student2 = np.mean(student2)
var_student2 = np.var(student2)


print(f"Teacher - Media: {mean_teacher}, Varianza: {var_teacher}")
#print(f"Student 1 - Media: {mean_student1}, Varianza: {var_student1}")
print(f"Student 2 - Media: {mean_student2}, Varianza: {var_student2}")

In [ ]:
def cumulative_mean(vector):
    cumsum = np.cumsum(vector)  #
    return cumsum / np.arange(1, len(vector) + 1)  

#evaluate the cumulative mean for each vector
mean_teacher = cumulative_mean(teacher)
#mean_student1 = cumulative_mean(student1)
mean_student2 = cumulative_mean(student2)


plt.figure(figsize=(8, 5))

plt.plot(mean_teacher, label='Teacher', marker='o')
#plt.plot(mean_student1, label='Student 1', marker='x')
plt.plot(mean_student2, label='Student 2', marker='^')


plt.xlabel('Episodes')
plt.ylabel('Mean Reward')
plt.legend()
plt.grid(False)


plt.show()